# Stage C — bounded genomic pilot
Trains separately initialized adaptive, reference, frozen-memory, and no-memory runs. Re-running resumes each run from Drive.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='0ae054f4c7622f4bffea4bb4c51b28abcf0db6e3'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
PILOT_NAME='c7_bounded_100mbp'
HORIZON=3
VALID_BASE_BUDGET=100_000_000
CHECKPOINT_EVERY=100

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess,sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
root=f'{DRIVE_ROOT}/runs/{PILOT_NAME}'
subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',root,'--label','hardware_preflight','--repo',str(repo),'--','seqtrainer-titans-stage-c-hardware-preflight','--require','A100'],check=True)

In [ ]:
for mode in ['adaptive','reference','frozen_memory','no_memory']:
    run_dir=f'{root}/{mode}'
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',DATASET_DIR,'--run-dir',run_dir,'--memory-mode',mode,'--horizon',str(HORIZON),'--batch-size','1','--max-valid-bases',str(VALID_BASE_BUDGET),'--checkpoint-every',str(CHECKPOINT_EVERY),'--activation','float32']
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',run_dir,'--label',f'train_{mode}','--repo',str(repo),'--',*command],check=True)
print('SHARE THIS DIRECTORY:',root)